# 1a — Polar to Cartesian: Turning LiDAR Rays into Points

A 2D LiDAR spins a laser around and, for each direction, reports how far away
the nearest obstacle is. The raw output is therefore a list of
**(angle, range)** pairs — *polar* coordinates centred on the sensor.

Every algorithm later in this curriculum (scan matching, occupancy grids, ICP)
works with **(x, y)** points instead. This notebook covers the conversion
between the two.

**Learning goals**

1. See what a raw LiDAR scan looks like.
2. Convert polar readings to Cartesian points with `cos` and `sin`.
3. Convert back with `arctan2` and `hypot`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## A synthetic 2D LiDAR

Rather than use a real sensor we simulate one. The environment is a list of
wall segments; for each ray we find the closest segment it hits.
The maths is a standard ray–segment intersection — it's not important to
follow every line, but it's useful to know the simulator is only a few lines
of NumPy and not a black box.

In [ ]:
def cast_ray(origin, angle, segments, max_range):
    """Distance along a ray from `origin` at `angle` to the nearest wall segment.
    Returns `max_range` if nothing is hit."""
    d = np.array([np.cos(angle), np.sin(angle)])
    best = max_range
    for p, q in segments:
        p, q = np.asarray(p, float), np.asarray(q, float)
        e = q - p
        denom = d[0] * e[1] - d[1] * e[0]
        if abs(denom) < 1e-12:          # ray parallel to this wall
            continue
        w = p - origin
        t = (w[0] * e[1] - w[1] * e[0]) / denom   # distance along the ray
        u = (w[0] * d[1] - w[1] * d[0]) / denom   # position along the segment (0..1)
        if t >= 0 and 0 <= u <= 1 and t < best:
            best = t
    return best


def lidar_scan(pose, segments, num_rays=360, max_range=10.0):
    """Simulate a 2D LiDAR at `pose = (x, y, theta)`.
    Returns (angles, ranges) in the *robot* frame; angle 0 is the robot's +x axis."""
    x, y, theta = pose
    angles = np.linspace(-np.pi, np.pi, num_rays, endpoint=False)
    ranges = np.array([cast_ray(np.array([x, y]), theta + a, segments, max_range) for a in angles])
    return angles, ranges


def polar_to_cartesian(angles, ranges):
    """Convert (angle, range) pairs to an (N, 2) array of (x, y) points."""
    return np.column_stack((ranges * np.cos(angles), ranges * np.sin(angles)))

In [ ]:
# A simple room: a 10 m x 8 m rectangle with a pillar in one corner.
room = [
    ((-5, -4), (5, -4)),   # bottom wall
    ((5, -4), (5, 4)),     # right wall
    ((5, 4), (-5, 4)),     # top wall
    ((-5, 4), (-5, -4)),   # left wall
    ((2, 1), (3, 1)), ((3, 1), (3, 2)), ((3, 2), (2, 2)), ((2, 2), (2, 1)),  # pillar
]

angles, ranges = lidar_scan(pose=(0.0, 0.0, 0.0), segments=room, num_rays=180)
print("angles (rad):", np.round(angles[:5], 3), "...")
print("ranges (m):  ", np.round(ranges[:5], 3), "...")

## The raw data: ranges and bearings

Plotting the readings on a polar axis shows exactly what the sensor
"sees": one distance per direction, with the sensor at the centre.

In [ ]:
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection="polar")
ax.scatter(angles, ranges, s=8)
ax.set_title("Raw LiDAR scan (angle vs. range)")
plt.show()

## The conversion

A point at distance $r$ along direction $\theta$ has coordinates

$$x = r\cos\theta, \qquad y = r\sin\theta .$$

That's the whole conversion. Stacking the results gives the `(N, 2)` array
of points used everywhere else in these tutorials.

In [ ]:
points = polar_to_cartesian(angles, ranges)
print("points shape:", points.shape)
print(points[:3])

In [ ]:
fig, (ax_polar, ax_xy) = plt.subplots(1, 2, figsize=(12, 6))

ax_polar.set_title("Polar: one range per angle")
ax_polar.plot(np.degrees(angles), ranges, ".-", ms=4)
ax_polar.set_xlabel("bearing (degrees)")
ax_polar.set_ylabel("range (m)")
ax_polar.grid(True)

ax_xy.set_title("Cartesian: the same scan as (x, y) points")
for (x, y) in points:                      # draw every ray from the sensor
    ax_xy.plot([0, x], [0, y], color="orange", lw=0.3, alpha=0.6)
ax_xy.scatter(points[:, 0], points[:, 1], s=10, c="blue", label="scan points")
ax_xy.scatter(0, 0, marker="^", s=120, c="red", label="sensor")
for p, q in room:
    ax_xy.plot([p[0], q[0]], [p[1], q[1]], "k-", lw=1, alpha=0.4)
ax_xy.set_xlabel("x (m)")
ax_xy.set_ylabel("y (m)")
ax_xy.axis("equal")
ax_xy.legend()
plt.show()

## Watching the conversion ray by ray

Move the slider (or change `n` by hand) to convert the first `n` rays only.
Each orange ray is one `(angle, range)` reading; the blue dot at its tip is
the resulting `(x, y)` point.

In [ ]:
def show_first_n(n=20):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_title(f"First {n} of {len(angles)} rays converted")
    for (x, y) in points[:n]:
        ax.plot([0, x], [0, y], color="orange", lw=0.8)
    ax.scatter(points[:n, 0], points[:n, 1], s=15, c="blue")
    ax.scatter(0, 0, marker="^", s=120, c="red")
    for p, q in room:
        ax.plot([p[0], q[0]], [p[1], q[1]], "k-", lw=1, alpha=0.4)
    ax.set_xlim(-6, 6); ax.set_ylim(-5, 5); ax.set_aspect("equal")
    plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(show_first_n, n=IntSlider(min=1, max=len(angles), step=1, value=20));
except ImportError:          # ipywidgets not installed: just draw one frame
    show_first_n(20)

## Going back: Cartesian → polar

Sometimes we need the reverse (for example to re-render a map as a
predicted scan). The inverse uses `hypot` for the range and `arctan2` for
the angle — **not** `arctan`, which loses the quadrant.

In [ ]:
def cartesian_to_polar(points):
    r = np.hypot(points[:, 0], points[:, 1])
    theta = np.arctan2(points[:, 1], points[:, 0])
    return theta, r

theta_back, r_back = cartesian_to_polar(points)
print("max angle error:", np.abs(theta_back - angles).max())
print("max range error:", np.abs(r_back - ranges).max())

## Summary

* A LiDAR scan is a list of `(angle, range)` pairs in the sensor's own frame.
* `x = r cos θ`, `y = r sin θ` converts each reading into a point; we keep
  scans as `(N, 2)` NumPy arrays from here on.
* `arctan2` / `hypot` convert back exactly.

**Next:** [1b — Matrix transforms](1b_matrix_transforms.ipynb) shows how to
move and rotate a whole scan at once, which is what a scan matcher searches over.